In [0]:
"""
Trains the trip propensity module. Read in transformed data from
ETL.py. Read in features to train on from s3

Trains the model in two iterations - first iteration on everyone,
second iteration on 10% of the members with >0.95 and <0.05 probabilty
of making a trip and everybody in the middle.
"""

In [0]:
%run ../../config/utils

In [0]:
from datetime import datetime

import pandas as pd
import yaml
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import sys
sys.path.append('..')
sys.path.append('../..')

import lib_trip_spend.python_general_utilities as util_func
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.client import MlflowClient
import matplotlib.pyplot as plt
import shap
import numpy as np

In [0]:
with open('./config/config.yml', "r") as stream:
    config = yaml.load(stream, Loader=yaml.FullLoader)

START_WINDOW = config["trip"]["start_window"] - 1
END_WINDOW = config["trip"]["end_window"] - 1
SPLIT_RATE = config["shared"]["split_rate"]
SAMPLE_RATE = config["shared"]["sample_rate"]
BUCKET = config["shared"]["bucket"]
NUMBER_OF_TREES = config["trip"]["number_of_trees"]
MAX_DEPTH = config["trip"]["max_depth"]
MAX_FEATURES = config["trip"]["max_features"]
MIN_LEAF_SIZE = config["trip"]["min_sample_leaf"]
MODEL = config["trip"]["model"]
DATE = datetime.today().strftime("%Y%m%d")


run_name = config["shared"]["run_name"]
CLASSIFICATION_METRICS = config["trip"]["metrics"]

In [0]:
def create_rf_model(
    training_X,
    training_y,
    testing_X,
    max_depth,
    max_features,
    n_estimators,
    min_samples_leaf,
):
    model = RandomForestClassifier(
        n_jobs=-1,
        max_depth=max_depth,
        max_features=max_features,
        warm_start=True,
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
    )

    model.fit(training_X, training_y.values.ravel())
    return (
        model,
        model.predict(testing_X),
        model.feature_importances_,
        model.predict_proba(testing_X)[:, 1],
    )


def create_lr_model(training_X, training_y, testing_X):
    training_X = training_X[training_X.columns.intersection(features)]
    testing_X = testing_X[training_X.columns.intersection(features)]
    model = LogisticRegression(n_jobs=-1)
    model.fit(training_X, training_y.values.ravel())
    return (
        model,
        model.predict(testing_X),
        model.coef_,
        model.predict_proba(testing_X)[:, 1],
    )

In [0]:
features = pd.read_csv(feature_path)

label_column = "will_visit_from_%s_%s" % (str(START_WINDOW), str(END_WINDOW))
(
    column_headers,
    categorical_features,
    continious_features,
) = util_func.get_column_headers(features, [label_column])

In [0]:
data = spark.table(model_trip_spend_etl_output).select(*column_headers) 

In [0]:
training, testing = util_func.test_and_train_to_pandas(
    data.toPandas(), SPLIT_RATE, SAMPLE_RATE
)

In [0]:
experiment_name = experiment_name_trip_propensity

mlflow.sklearn.autolog(log_input_examples=True, log_models=False, log_datasets=False)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

feature_dataset = mlflow.data.from_spark(data, name = 'trips_propensity_etl_dataset')

In [0]:
%sh
mkdir tmp

In [0]:
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')
with mlflow.start_run(run_name=f'training_{run_name}_{mark_datetime}') as run:
    mlflow.log_input(feature_dataset, context="source")
    mlflow.log_input(mlflow.data.from_pandas(training, source=feature_dataset.source), context="training_1")
    mlflow.log_input(mlflow.data.from_pandas(testing, source=feature_dataset.source), context="testing_1")

    if MODEL == "rf":
        model, predictions, feature_importance, prediction_prob = create_rf_model(
            training.drop(columns=[label_column]),
            training[[label_column]],
            testing.drop(columns=[label_column]),
            MAX_DEPTH,
            MAX_FEATURES,
            NUMBER_OF_TREES,
            MIN_LEAF_SIZE,
        )
    if MODEL == "lr":
        model, predictions, feature_importance, prediction_prob = create_lr_model(
            training.drop(columns=[label_column]),
            training[[label_column]],
            testing.drop(columns=[label_column]),
        )
    print("---------------------completed model creation--------------------")
    model_metrics = [
        util_func.create_trip_propensity_metric(
            predictions, testing[[label_column]]
        )
    ]

    combined = training.append(testing)
    if MODEL == "lr":
        combined = combined[
            combined.columns.intersection(features + [label_column])
        ]

    pred = model.predict_proba(combined.drop(columns=[label_column]))
    combined["prediction"] = pred[:, 1]
    print(
        "----------------initial length of combined data {}---------------".format(
            len(combined)
        )
    )
    high_predicted_value = combined.loc[combined["prediction"] > 0.95].sample(
        frac=0.1, replace=False
    )
    low_predicted_value = combined.loc[combined["prediction"] < 0.05].sample(
        frac=0.1, replace=False
    )

    combined = combined.loc[
        (combined["prediction"] >= 0.05) & (combined["prediction"] <= 0.95)
    ]

    combined = combined.append([high_predicted_value, low_predicted_value])
    print(
        "----------------data for second iteration after filtering {}---------------".format(
            len(combined)
        )
    )
    testing["prediction"] = prediction_prob

    combined = combined.drop(columns=["prediction"])
    print("---------------------running second iteration---------------------")
    #  filter on predictions and rerun model
    if len(combined) != 0:
        training, testing = train_test_split(combined, test_size=SPLIT_RATE)
        mlflow.log_input(mlflow.data.from_pandas(training, source=feature_dataset.source), context="training_2")
        mlflow.log_input(mlflow.data.from_pandas(testing, source=feature_dataset.source), context="testing_2")
        model, predictions, feature_importance, prediction_prob = create_rf_model(
            training.drop(columns=[label_column]),
            training[[label_column]],
            testing.drop(columns=[label_column]),
            MAX_DEPTH,
            MAX_FEATURES,
            NUMBER_OF_TREES,
            MIN_LEAF_SIZE,
        )
        model_metrics += [
        util_func.create_trip_propensity_metric(
            predictions, testing[[label_column]]
        )
        ]
        second_iteration_output_df = testing[[label_column]]

        second_iteration_output_df["prediction"] = predictions
        second_iteration_output_df["prediction_prob"] = prediction_prob



    model_metrics = util_func.get_df_from_list(
        model_metrics, CLASSIFICATION_METRICS
        )
    feature_importances_pd = pd.DataFrame(
            util_func.create_sklearn_features(feature_importance, column_headers)
        )
    model_metrics.to_csv(trip_metrics_path, index=False)
    feature_importances_pd.to_csv(trip_feature_importance_path, index=False)
    mlflow.log_artifact(trip_metrics_path)
    mlflow.log_artifact(trip_feature_importance_path)

    signature = infer_signature(training.drop(columns=[label_column]), training[[label_column]].values.ravel())

    model_info = mlflow.sklearn.log_model(
            sk_model = model,
            artifact_path = "model",
            signature = signature,
            registered_model_name = trip_propensity_model_catalog
        )


    print("---------------------starting manual SHAP plot generation---------------------")


    X_test_for_shap = testing.drop(columns=[label_column, "prediction"], errors='ignore').sample(n=min(5000, len(testing)))

    classes = getattr(model, "classes_", np.array([0, 1]))
    pos_label = 1
    pos_idx = int(np.where(classes == pos_label)[0][0])

    # choose explainer
    if MODEL == "rf":
        explainer = shap.TreeExplainer(model)
        sv_raw = explainer.shap_values(X_test_for_shap)
    elif MODEL == "lr":
        explainer = shap.LinearExplainer(model, X_test_for_shap, feature_perturbation="interventional")
        sv_raw = explainer.shap_values(X_test_for_shap)

    if isinstance(sv_raw, list):
        sv = sv_raw[pos_idx]                       # list of arrays per class
    elif isinstance(sv_raw, np.ndarray) and sv_raw.ndim == 3:
        sv = sv_raw[..., pos_idx]                  # last axis is classes
    else:
        sv = sv_raw   

    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, X_test_for_shap, show=False)
    plt.tight_layout()
    plt.savefig("./tmp/shap_summary_plot.png"); plt.close()

    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, X_test_for_shap, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig("./tmp/shap_feature_importance_plot.png"); plt.close()

    mlflow.log_artifact("./tmp/shap_summary_plot.png", artifact_path="shap_plots")
    mlflow.log_artifact("./tmp/shap_feature_importance_plot.png", artifact_path="shap_plots")
    mlflow.log_artifact("./config/config.yml")

In [0]:
client = MlflowClient()
client.set_registered_model_alias(trip_propensity_model_catalog, "champion", model_info.registered_model_version)

In [0]:
%sh
rm -r tmp